In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/telco_customer_churn_cleaned.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df = df.drop("customerID", axis=1)

In [5]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [6]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [7]:
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (7032, 19)
Target Shape   : (7032,)


In [8]:
y.value_counts()

Churn
No     5163
Yes    1869
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print("Training Target   :", y_train.shape)
print("Testing Target    :", y_test.shape)

Training Features : (5625, 19)
Testing Features  : (1407, 19)
Training Target   : (5625,)
Testing Target    : (1407,)


In [11]:
print("Original Dataset")
print(y.value_counts(normalize=True))

print("\nTraining Set")
print(y_train.value_counts(normalize=True))

print("\nTesting Set")
print(y_test.value_counts(normalize=True))

Original Dataset
Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64

Training Set
Churn
No     0.734222
Yes    0.265778
Name: proportion, dtype: float64

Testing Set
Churn
No     0.734186
Yes    0.265814
Name: proportion, dtype: float64


In [12]:
y_train = y_train.map({
    "No": 0,
    "Yes": 1
})

y_test = y_test.map({
    "No": 0,
    "Yes": 1
})

In [13]:
print(y_train.value_counts())
print()

print(y_test.value_counts())

Churn
0    4130
1    1495
Name: count, dtype: int64

Churn
0    1033
1     374
Name: count, dtype: int64


In [14]:
for column in X_train.columns:
    print(f"{column}: {X_train[column].nunique()} unique values")

gender: 2 unique values
SeniorCitizen: 2 unique values
Partner: 2 unique values
Dependents: 2 unique values
tenure: 72 unique values
PhoneService: 2 unique values
MultipleLines: 3 unique values
InternetService: 3 unique values
OnlineSecurity: 3 unique values
OnlineBackup: 3 unique values
DeviceProtection: 3 unique values
TechSupport: 3 unique values
StreamingTV: 3 unique values
StreamingMovies: 3 unique values
Contract: 3 unique values
PaperlessBilling: 2 unique values
PaymentMethod: 4 unique values
MonthlyCharges: 1500 unique values
TotalCharges: 5294 unique values


In [15]:
binary_columns = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

In [16]:
binary_mapping = {
    "Yes": 1,
    "No": 0,
    "Male": 1,
    "Female": 0
}

In [17]:
X_train[binary_columns] = X_train[binary_columns].replace(binary_mapping)

In [18]:
X_test[binary_columns] = X_test[binary_columns].replace(binary_mapping)

In [19]:
X_train[binary_columns].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling
1408,1,1,1,1,0
6992,1,0,0,0,0
3349,0,1,0,1,0
4486,1,0,0,1,0
3535,0,1,0,0,0


In [20]:
from sklearn.preprocessing import OneHotEncoder

In [21]:
categorical_columns = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

In [22]:
import joblib

joblib.dump(
    categorical_columns,
    "../models/categorical_columns.pkl"
)

print("Categorical columns saved!")

Categorical columns saved!


In [23]:
encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

In [24]:
encoded_train = encoder.fit_transform(
    X_train[categorical_columns]
)

In [25]:
encoded_test = encoder.transform(
    X_test[categorical_columns]
)

In [26]:
import joblib

joblib.dump(
    encoder,
    "../models/encoder.pkl"
)

print("Encoder saved!")

Encoder saved!


In [27]:
encoded_columns = encoder.get_feature_names_out(categorical_columns)

In [28]:
encoded_train = pd.DataFrame(
    encoded_train,
    columns=encoded_columns,
    index=X_train.index
)

encoded_test = pd.DataFrame(
    encoded_test,
    columns=encoded_columns,
    index=X_test.index
)

In [29]:
X_train = X_train.drop(columns=categorical_columns)
X_test = X_test.drop(columns=categorical_columns)

In [30]:
X_train = pd.concat([X_train, encoded_train], axis=1)
X_test = pd.concat([X_test, encoded_test], axis=1)

In [31]:
print(X_train.head())
print()
print(X_train.shape)

     gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
1408      1              0       1          1      65            1   
6992      1              0       0          0      26            0   
3349      0              0       1          0      68            1   
4486      1              0       0          0       3            1   
3535      0              0       1          0      49            0   

     PaperlessBilling  MonthlyCharges  TotalCharges  \
1408                0           94.55       6078.75   
6992                0           35.75       1022.50   
3349                0           90.20       6297.65   
4486                0           84.30        235.05   
3535                0           40.65       2070.75   

      MultipleLines_No phone service  MultipleLines_Yes  \
1408                             0.0                1.0   
6992                             1.0                0.0   
3349                             0.0                1.0   
4486        

In [32]:
from sklearn.preprocessing import StandardScaler

In [33]:
scaler = StandardScaler()

In [34]:
numerical_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [35]:
X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

In [36]:
joblib.dump(scaler, "../models/scaler.pkl")
print("Scaler saved!")

Scaler saved!


In [37]:
X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [38]:
X_train[numerical_columns].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,5625.00,5625.00,5625.00
mean,-0.00,0.00,0.00
std,1.00,1.00,1.00
min,-1.29,-1.55,-1.00
25%,-0.96,-0.97,-0.83
50%,-0.15,0.19,-0.39
75%,0.96,0.83,0.66
max,1.61,1.78,2.81


In [39]:
# Save processed datasets

X_train.to_csv("../data/final/X_train.csv", index=False)
X_test.to_csv("../data/final/X_test.csv", index=False)

y_train.to_csv("../data/final/y_train.csv", index=False)
y_test.to_csv("../data/final/y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
